# Xiangqi AI — Tabular Q-Learning (Gold v2)

Huấn luyện **Tabular Q-Learning offline** trên Gold `reward_v2_shaped`. Chỉ dùng train để cập nhật Q; val/test chỉ để đánh giá overlap. Đọc Gold Drive ở chế độ read-only. Copy Gold về `/content` trước khi train, sau đó lưu Q-table và report vào một model run mới trên Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/XiangqiAI")
GOLD_DIR = DRIVE_ROOT / "data" / "03_gold" / "reward_v2_shaped"
MODEL_ROOT = DRIVE_ROOT / "models" / "tabular_qlearning"
GITHUB_REPO = "https://github.com/Ko4rut/ML-Advance-Group.git"
GIT_BRANCH = "feature/tabular-qlearning-gold-v2"
ALPHA = 0.2
GAMMA = 0.99
EPOCHS = 12
SEED = 42

In [ ]:
import subprocess
import sys
import tempfile
workspace = Path(tempfile.mkdtemp(prefix="xiangqi_tabular_", dir="/content"))
REPO_DIR = workspace / "ML-Advance-Group"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GITHUB_REPO, str(REPO_DIR)],
    check=True,
)
BACKEND_DIR = REPO_DIR / "project" / "backend"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)], check=True)
REVISION = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Git commit:", REVISION)

In [ ]:
import hashlib
import shutil
import json
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not GOLD_DIR.is_dir():
    raise FileNotFoundError(f"Missing Gold v2 folder: {GOLD_DIR}")
LOCAL_GOLD = Path(tempfile.mkdtemp(prefix="tabular_gold_data_", dir="/content"))
required = ["offline_rl_train.jsonl", "offline_rl_val.jsonl", "offline_rl_test.jsonl", "report.json"]
for filename in required:
    src = GOLD_DIR / filename
    dst = LOCAL_GOLD / filename
    if not src.is_file():
        raise FileNotFoundError(src)
    shutil.copy2(src, dst)
    if sha256_file(src) != sha256_file(dst):
        raise IOError(f"Gold copy checksum mismatch: {filename}")
report_gold = json.loads((LOCAL_GOLD / "report.json").read_text(encoding="utf-8"))
for filename in required[:-1]:
    if report_gold.get("output_sha256", {}).get(filename) != sha256_file(LOCAL_GOLD / filename):
        raise IOError(f"Gold report SHA-256 mismatch: {filename}")
print("Gold inputs validated:", report_gold["transitions"], "transitions")

In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_NAME = f"tabular_q_gold_v2_seed{SEED}_{stamp}"
LOCAL_OUTPUT = Path("/content/xiangqi_tabular_models") / RUN_NAME
command = [
    sys.executable, "-m", "scripts.train_tabular_qlearning",
    "--gold-dir", str(LOCAL_GOLD),
    "--output-dir", str(LOCAL_OUTPUT),
    "--alpha", str(ALPHA), "--gamma", str(GAMMA),
    "--epochs", str(EPOCHS), "--seed", str(SEED),
]
subprocess.run(command, cwd=BACKEND_DIR, check=True)
TRAIN_REPORT = json.loads((LOCAL_OUTPUT / "report.json").read_text(encoding="utf-8"))
print("Known states:", TRAIN_REPORT["known_states"])
print("Known state-actions:", TRAIN_REPORT["known_state_action_pairs"])
print("Final epoch:", TRAIN_REPORT["history"][-1])
print("Validation:", TRAIN_REPORT["splits"]["val"])
print("Test:", TRAIN_REPORT["splits"]["test"])

In [ ]:
# Publish checkpoint in NEW run folder. Never overwrite Gold or existing models.
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
target = MODEL_ROOT / RUN_NAME
staging = MODEL_ROOT / f".{RUN_NAME}.uploading"
if target.exists() or staging.exists():
    raise FileExistsError(f"Model output already exists: {RUN_NAME}")
staging.mkdir(parents=True, exist_ok=False)
for src in LOCAL_OUTPUT.iterdir():
    if not src.is_file():
        continue
    dst = staging / src.name
    shutil.copy2(src, dst)
    if sha256_file(src) != sha256_file(dst):
        raise IOError(f"Model copy checksum mismatch: {src.name}")
staging.rename(target)
print("Published:", target)
print("Git revision:", REVISION)